# 한이음 NLU Structure B 학습
## koELECTRA + Item Query Transformer Decoder

이 노트북은 `merged_structure_b_with_status_and_synthetic.jsonl`을 학습하여 다음을 동시에 예측합니다.

- `intent`
- `order_status`
- 주문 항목 수
- 각 주문 항목의 `menu`, `quantity`, `temperature`

모델 구조:

```text
koELECTRA encoder
        ↓
CLS → intent / order_status heads
        ↓
learned item queries
        ↓
Transformer decoder
        ↓
item active / menu / quantity / temperature heads
```

특징:

- 최대 3개 주문 항목 지원
- 빈 슬롯은 `NONE` 클래스로 학습
- 기존 train/valid/test 출처를 유지
- synthetic 데이터는 train/valid/test에 8:1:1로 분리
- validation `frame_accuracy` 기준으로 최적 모델 저장
- 마지막 셀에서 평가 CSV 자동 생성 및 다운로드

## 1. 패키지 설치

In [ ]:
%pip -q install -U "transformers==4.46.3" "scikit-learn>=1.4" "pandas>=2.0" "tqdm>=4.66"

## 2. 환경 설정

In [ ]:
import json
import math
import os
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 42
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
MAX_LENGTH = 96
MAX_ITEMS = 3
BATCH_SIZE = 8
EPOCHS = 15
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("device:", DEVICE)

## 3. 데이터 불러오기

아래 셀은 두 방식 중 하나를 지원합니다.

1. Colab 파일 업로드
2. Google Drive 또는 GitHub에서 내려받은 로컬 경로 직접 지정

파일 이름은 기본적으로 `merged_structure_b_with_status_and_synthetic.jsonl`을 사용합니다.

In [ ]:
DATA_PATH = Path("/content/merged_structure_b_v3.jsonl")

if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("JSONL 파일을 업로드해주세요.")
    uploaded_name = next(iter(uploaded))
    DATA_PATH = Path("/content") / uploaded_name

print("dataset:", DATA_PATH)

## 4. JSONL 로드 및 데이터 분할

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{line_number}번째 줄 JSON 오류: {exc}") from exc
    return records

records = load_jsonl(DATA_PATH)
print("total:", len(records))

original_train, original_valid, original_test, synthetic = [], [], [], []
for record in records:
    split = str((record.get("source") or {}).get("split") or "").lower()
    if split == "train":
        original_train.append(record)
    elif split == "valid":
        original_valid.append(record)
    elif split == "test":
        original_test.append(record)
    elif split == "synthetic" or record.get("synthetic"):
        synthetic.append(record)
    else:
        original_train.append(record)

rng = random.Random(SEED)
rng.shuffle(synthetic)
syn_train_end = int(len(synthetic) * 0.8)
syn_valid_end = int(len(synthetic) * 0.9)

train_records = original_train + synthetic[:syn_train_end]
valid_records = original_valid + synthetic[syn_train_end:syn_valid_end]
test_records = original_test + synthetic[syn_valid_end:]

rng.shuffle(train_records)
rng.shuffle(valid_records)
rng.shuffle(test_records)

print({
    "train": len(train_records),
    "valid": len(valid_records),
    "test": len(test_records),
    "synthetic_total": len(synthetic),
})

## 5. 라벨 맵 생성

In [ ]:
INTENT_LABELS = sorted({str(record.get("intent") or "UNKNOWN") for record in records})
STATUS_LABELS = ["NONE", "VALID", "INCOMPLETE", "CONFLICT", "OUT_OF_POLICY", "UNPARSABLE"]

# 실제 메뉴판 기준
MENU_LABELS = [
    "NONE",
    "아메리카노",
    "카페라떼",
    "바닐라라떼",
    "레몬에이드",
    "딸기스무디",
]

TEMPERATURE_LABELS = ["NONE", "ICE", "HOT"]
QUANTITY_LABELS = ["NONE"] + [str(value) for value in range(1, 21)]

def make_maps(labels: list[str]):
    label2id = {label: index for index, label in enumerate(labels)}
    id2label = {index: label for label, index in label2id.items()}
    return label2id, id2label

intent2id, id2intent = make_maps(INTENT_LABELS)
status2id, id2status = make_maps(STATUS_LABELS)
menu2id, id2menu = make_maps(MENU_LABELS)
temperature2id, id2temperature = make_maps(TEMPERATURE_LABELS)
quantity2id, id2quantity = make_maps(QUANTITY_LABELS)

LABEL_MAPS = {
    "intent": {"label2id": intent2id, "id2label": id2intent},
    "order_status": {"label2id": status2id, "id2label": id2status},
    "menu": {"label2id": menu2id, "id2label": id2menu},
    "temperature": {"label2id": temperature2id, "id2label": id2temperature},
    "quantity": {"label2id": quantity2id, "id2label": id2quantity},
}

print(json.dumps({
    "intent": INTENT_LABELS,
    "order_status": STATUS_LABELS,
    "menu": MENU_LABELS,
    "temperature": TEMPERATURE_LABELS,
    "quantity": QUANTITY_LABELS,
}, ensure_ascii=False, indent=2))

## 6. 데이터 정규화 및 Dataset

In [ ]:
def normalize_status(record: dict[str, Any]) -> str:
    if str(record.get("intent")) != "ORDER":
        return "NONE"
    status = str(record.get("order_status") or "UNPARSABLE")
    return status if status in status2id else "UNPARSABLE"

def normalize_menu(value: Any) -> str:
    text = str(value).strip() if value is not None else "NONE"
    aliases = {
        "라떼": "카페라떼",
        "카페라테": "카페라떼",
        "바닐라 라떼": "바닐라라떼",
        "바닐라라테": "바닐라라떼",
    }
    text = aliases.get(text, text)
    return text if text in menu2id else "NONE"

def normalize_temperature(value: Any) -> str:
    text = str(value).strip().upper() if value is not None else "NONE"
    return text if text in temperature2id else "NONE"

def normalize_quantity(value: Any) -> str:
    if value is None:
        return "NONE"
    try:
        number = int(float(value))
        return str(number) if 1 <= number <= 20 else "NONE"
    except (TypeError, ValueError):
        return "NONE"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class StructureBDataset(Dataset):
    def __init__(self, records: list[dict[str, Any]]):
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        encoded = tokenizer(
            str(record.get("text") or ""),
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        item_active = [0] * MAX_ITEMS
        menu_labels = [menu2id["NONE"]] * MAX_ITEMS
        temperature_labels = [temperature2id["NONE"]] * MAX_ITEMS
        quantity_labels = [quantity2id["NONE"]] * MAX_ITEMS

        items = list(record.get("items") or [])[:MAX_ITEMS]
        for item_index, item in enumerate(items):
            item_active[item_index] = 1
            menu_labels[item_index] = menu2id[normalize_menu(item.get("menu"))]
            temperature_labels[item_index] = temperature2id[
                normalize_temperature(item.get("temperature"))
            ]
            quantity_labels[item_index] = quantity2id[
                normalize_quantity(item.get("quantity"))
            ]

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "intent_label": torch.tensor(
                intent2id.get(str(record.get("intent")), intent2id["UNKNOWN"]),
                dtype=torch.long,
            ),
            "status_label": torch.tensor(
                status2id[normalize_status(record)],
                dtype=torch.long,
            ),
            "item_active": torch.tensor(item_active, dtype=torch.long),
            "menu_labels": torch.tensor(menu_labels, dtype=torch.long),
            "temperature_labels": torch.tensor(temperature_labels, dtype=torch.long),
            "quantity_labels": torch.tensor(quantity_labels, dtype=torch.long),
            "record_index": torch.tensor(index, dtype=torch.long),
        }

train_dataset = StructureBDataset(train_records)
valid_dataset = StructureBDataset(valid_records)
test_dataset = StructureBDataset(test_records)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 7. koELECTRA + Item Query Transformer Decoder

In [ ]:
class ItemQueryDecoderModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.encoder.config.hidden_size

        self.intent_head = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(hidden_size, len(INTENT_LABELS)),
        )
        self.status_head = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(hidden_size, len(STATUS_LABELS)),
        )

        self.item_queries = nn.Parameter(torch.randn(MAX_ITEMS, hidden_size) * 0.02)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=4,
            dim_feedforward=hidden_size * 4,
            dropout=0.15,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.item_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.item_norm = nn.LayerNorm(hidden_size)

        self.active_head = nn.Linear(hidden_size, 2)
        self.menu_head = nn.Linear(hidden_size, len(MENU_LABELS))
        self.temperature_head = nn.Linear(hidden_size, len(TEMPERATURE_LABELS))
        self.quantity_head = nn.Linear(hidden_size, len(QUANTITY_LABELS))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        memory = outputs.last_hidden_state
        cls = memory[:, 0]

        batch_size = input_ids.size(0)
        queries = self.item_queries.unsqueeze(0).expand(batch_size, -1, -1)
        decoded = self.item_decoder(
            tgt=queries,
            memory=memory,
            memory_key_padding_mask=(attention_mask == 0),
        )
        decoded = self.item_norm(decoded)

        return {
            "intent_logits": self.intent_head(cls),
            "status_logits": self.status_head(cls),
            "active_logits": self.active_head(decoded),
            "menu_logits": self.menu_head(decoded),
            "temperature_logits": self.temperature_head(decoded),
            "quantity_logits": self.quantity_head(decoded),
        }

model = ItemQueryDecoderModel().to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. 손실 함수

In [ ]:
ce_loss = nn.CrossEntropyLoss()

def compute_loss(outputs: dict[str, torch.Tensor], batch: dict[str, torch.Tensor]):
    intent_loss = ce_loss(outputs["intent_logits"], batch["intent_label"])
    status_loss = ce_loss(outputs["status_logits"], batch["status_label"])

    active_loss = ce_loss(
        outputs["active_logits"].reshape(-1, 2),
        batch["item_active"].reshape(-1),
    )

    active_mask = batch["item_active"].bool().reshape(-1)
    if active_mask.any():
        menu_loss = ce_loss(
            outputs["menu_logits"].reshape(-1, len(MENU_LABELS))[active_mask],
            batch["menu_labels"].reshape(-1)[active_mask],
        )
        temperature_loss = ce_loss(
            outputs["temperature_logits"].reshape(-1, len(TEMPERATURE_LABELS))[active_mask],
            batch["temperature_labels"].reshape(-1)[active_mask],
        )
        quantity_loss = ce_loss(
            outputs["quantity_logits"].reshape(-1, len(QUANTITY_LABELS))[active_mask],
            batch["quantity_labels"].reshape(-1)[active_mask],
        )
    else:
        zero = outputs["intent_logits"].sum() * 0.0
        menu_loss = temperature_loss = quantity_loss = zero

    total_loss = (
        1.0 * intent_loss
        + 1.0 * status_loss
        + 1.0 * active_loss
        + 1.3 * menu_loss
        + 1.0 * temperature_loss
        + 1.0 * quantity_loss
    )
    return total_loss, {
        "intent": intent_loss.detach().item(),
        "status": status_loss.detach().item(),
        "active": active_loss.detach().item(),
        "menu": menu_loss.detach().item(),
        "temperature": temperature_loss.detach().item(),
        "quantity": quantity_loss.detach().item(),
    }

## 9. 평가 함수

In [ ]:
def move_batch(batch: dict[str, Any]) -> dict[str, Any]:
    return {
        key: value.to(DEVICE) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }

@torch.no_grad()
def evaluate(loader: DataLoader, collect_predictions: bool = False):
    model.eval()
    losses = []
    true_intents, pred_intents = [], []
    true_statuses, pred_statuses = [], []
    true_active_all, pred_active_all = [], []
    true_menu_all, pred_menu_all = [], []
    true_temp_all, pred_temp_all = [], []
    true_qty_all, pred_qty_all = [], []
    row_predictions = []

    for batch in tqdm(loader, leave=False):
        batch = move_batch(batch)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss, _ = compute_loss(outputs, batch)
        losses.append(loss.item())

        pred_intent = outputs["intent_logits"].argmax(-1)
        pred_status = outputs["status_logits"].argmax(-1)
        pred_active = outputs["active_logits"].argmax(-1)
        pred_menu = outputs["menu_logits"].argmax(-1)
        pred_temp = outputs["temperature_logits"].argmax(-1)
        pred_qty = outputs["quantity_logits"].argmax(-1)

        true_intents.extend(batch["intent_label"].cpu().tolist())
        pred_intents.extend(pred_intent.cpu().tolist())
        true_statuses.extend(batch["status_label"].cpu().tolist())
        pred_statuses.extend(pred_status.cpu().tolist())

        true_active_all.extend(batch["item_active"].cpu().tolist())
        pred_active_all.extend(pred_active.cpu().tolist())
        true_menu_all.extend(batch["menu_labels"].cpu().tolist())
        pred_menu_all.extend(pred_menu.cpu().tolist())
        true_temp_all.extend(batch["temperature_labels"].cpu().tolist())
        pred_temp_all.extend(pred_temp.cpu().tolist())
        true_qty_all.extend(batch["quantity_labels"].cpu().tolist())
        pred_qty_all.extend(pred_qty.cpu().tolist())

        if collect_predictions:
            for i in range(batch["input_ids"].size(0)):
                row_predictions.append({
                    "dataset_index": int(batch["record_index"][i].item()),
                    "pred_intent_id": int(pred_intent[i].item()),
                    "pred_status_id": int(pred_status[i].item()),
                    "pred_active": pred_active[i].cpu().tolist(),
                    "pred_menu": pred_menu[i].cpu().tolist(),
                    "pred_temperature": pred_temp[i].cpu().tolist(),
                    "pred_quantity": pred_qty[i].cpu().tolist(),
                })

    true_active = np.array(true_active_all)
    pred_active = np.array(pred_active_all)
    true_menu = np.array(true_menu_all)
    pred_menu = np.array(pred_menu_all)
    true_temp = np.array(true_temp_all)
    pred_temp = np.array(pred_temp_all)
    true_qty = np.array(true_qty_all)
    pred_qty = np.array(pred_qty_all)

    active_positions = true_active == 1
    item_active_accuracy = float((true_active == pred_active).mean())

    def masked_accuracy(true_values, pred_values):
        if not active_positions.any():
            return 0.0
        return float((true_values[active_positions] == pred_values[active_positions]).mean())

    frame_matches = []
    for row in range(len(true_intents)):
        match = (
            true_intents[row] == pred_intents[row]
            and true_statuses[row] == pred_statuses[row]
            and np.array_equal(true_active[row], pred_active[row])
        )
        if match:
            for item_index in range(MAX_ITEMS):
                if true_active[row, item_index] == 1:
                    if not (
                        true_menu[row, item_index] == pred_menu[row, item_index]
                        and true_temp[row, item_index] == pred_temp[row, item_index]
                        and true_qty[row, item_index] == pred_qty[row, item_index]
                    ):
                        match = False
                        break
        frame_matches.append(match)

    metrics = {
        "loss": float(np.mean(losses)),
        "intent_accuracy": accuracy_score(true_intents, pred_intents),
        "intent_macro_f1": f1_score(true_intents, pred_intents, average="macro", zero_division=0),
        "status_accuracy": accuracy_score(true_statuses, pred_statuses),
        "status_macro_f1": f1_score(true_statuses, pred_statuses, average="macro", zero_division=0),
        "item_active_accuracy": item_active_accuracy,
        "menu_accuracy": masked_accuracy(true_menu, pred_menu),
        "temperature_accuracy": masked_accuracy(true_temp, pred_temp),
        "quantity_accuracy": masked_accuracy(true_qty, pred_qty),
        "frame_accuracy": float(np.mean(frame_matches)),
    }
    return metrics, row_predictions

## 10. 학습

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

CHECKPOINT_DIR = Path("/content/structure_b_item_decoder")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pt"

best_frame_accuracy = -1.0
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    progress = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for batch in progress:
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            outputs = model(batch["input_ids"], batch["attention_mask"])
            loss, loss_parts = compute_loss(outputs, batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_losses.append(loss.item())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    valid_metrics, _ = evaluate(valid_loader)
    epoch_result = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **{f"valid_{key}": value for key, value in valid_metrics.items()},
    }
    history.append(epoch_result)
    print(json.dumps(epoch_result, ensure_ascii=False, indent=2))

    if valid_metrics["frame_accuracy"] > best_frame_accuracy:
        best_frame_accuracy = valid_metrics["frame_accuracy"]
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "max_items": MAX_ITEMS,
                "label_maps": LABEL_MAPS,
                "epoch": epoch,
                "valid_metrics": valid_metrics,
            },
            BEST_MODEL_PATH,
        )
        tokenizer.save_pretrained(CHECKPOINT_DIR / "tokenizer")
        print("best model saved:", BEST_MODEL_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("early stopping")
            break

pd.DataFrame(history).to_csv(CHECKPOINT_DIR / "training_history.csv", index=False, encoding="utf-8-sig")

## 11. train&validation error 비교

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Train Loss",
)
plt.plot(
    history_df["epoch"],
    history_df["valid_loss"],
    marker="o",
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train Loss vs Validation Loss")
plt.xticks(history_df["epoch"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 12. 최적 모델 로드 후 test 최종 평가

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_metrics, raw_predictions = evaluate(test_loader, collect_predictions=True)
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

with (CHECKPOINT_DIR / "test_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(test_metrics, file, ensure_ascii=False, indent=2)

## 13. 평가 CSV 생성

In [ ]:
def true_items_from_record(record: dict[str, Any]) -> list[dict[str, Any]]:
    return [
        {
            "menu": item.get("menu"),
            "temperature": item.get("temperature"),
            "quantity": item.get("quantity"),
        }
        for item in list(record.get("items") or [])[:MAX_ITEMS]
    ]

def decode_predicted_items(prediction: dict[str, Any]) -> list[dict[str, Any]]:
    items = []
    for item_index in range(MAX_ITEMS):
        if prediction["pred_active"][item_index] != 1:
            continue
        menu = id2menu[prediction["pred_menu"][item_index]]
        temperature = id2temperature[prediction["pred_temperature"][item_index]]
        quantity = id2quantity[prediction["pred_quantity"][item_index]]
        items.append({
            "menu": None if menu == "NONE" else menu,
            "temperature": None if temperature == "NONE" else temperature,
            "quantity": None if quantity == "NONE" else int(quantity),
        })
    return items

rows = []
for prediction in raw_predictions:
    record = test_records[prediction["dataset_index"]]
    true_items = true_items_from_record(record)
    pred_items = decode_predicted_items(prediction)

    true_intent = str(record.get("intent") or "UNKNOWN")
    pred_intent = id2intent[prediction["pred_intent_id"]]
    true_status = normalize_status(record)
    pred_status = id2status[prediction["pred_status_id"]]

    intent_correct = true_intent == pred_intent
    status_correct = true_status == pred_status
    items_correct = true_items == pred_items

    rows.append({
        "text": record.get("text", ""),
        "source_split": (record.get("source") or {}).get("split"),
        "synthetic_type": record.get("synthetic_type"),
        "true_intent": true_intent,
        "pred_intent": pred_intent,
        "intent_correct": intent_correct,
        "true_order_status": true_status,
        "pred_order_status": pred_status,
        "status_correct": status_correct,
        "true_items_json": json.dumps(true_items, ensure_ascii=False),
        "pred_items_json": json.dumps(pred_items, ensure_ascii=False),
        "items_exact_match": items_correct,
        "frame_exact_match": intent_correct and status_correct and items_correct,
    })

evaluation_df = pd.DataFrame(rows)
evaluation_csv = Path("/content/structure_b_test_predictions.csv")
evaluation_df.to_csv(evaluation_csv, index=False, encoding="utf-8-sig")

metrics_csv = Path("/content/structure_b_test_metrics.csv")
pd.DataFrame([test_metrics]).to_csv(metrics_csv, index=False, encoding="utf-8-sig")

print("prediction CSV:", evaluation_csv)
print("metrics CSV:", metrics_csv)
display(evaluation_df.head(20))

## 14. 평가 CSV 다운로드

In [ ]:
from google.colab import files

files.download("/content/structure_b_test_predictions.csv")
files.download("/content/structure_b_test_metrics.csv")

## 15. 모델 파일 압축 및 다운로드

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/content/structure_b_item_decoder_model",
    "zip",
    CHECKPOINT_DIR,
)
print(archive_path)
files.download(archive_path)